## Set Up

In [1]:
%pip install -qU langchain-community faiss-cpu pydantic langchain_openai langchain aiofiles p langchain-tavily

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 0.2.0 requires numpy<2.0.0,>=1.26.2; python_version >= "3.12", but you have numpy 2.3.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\lberm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import getpass
import os


def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}:")


_set_env("OPENAI_API_KEY")

## Loading in VectorStore


In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from pathlib import Path

# Load paht
HERE = Path.cwd()
ROOT = HERE.parent
VSTORE_PATH = ROOT / "vectorstores" / "module_vectorstore_csv"

# Load Vector Store
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = FAISS.load_local(
    str(VSTORE_PATH), embeddings, allow_dangerous_deserialization=True
)

## Retrieval Settings

We are going to test two different retrieval mechanism a similarity search


### Similarity Search

This is a basic example of a similarity search 


In [111]:
docs = vectorstore.similarity_search_with_score(
    query="I want to study thermodynamics specifically specific heat questions some piston based questions only ",
    k=3,
)
docs

[(Document(id='29013f84-d589-41ab-97f1-8c506d2ea316', metadata={'source': 'src\\data\\QuestionDataV2_06122025_classified.csv', 'index': 229, 'relevant_courses': 'ME 100A, ME 100B', 'topics': 'Ideal Gas Behavior, Heat Transfer Area'}, page_content='5 kg of air fills the cylinder of a piston-cylinder assembly. The initial volume and pressure are 1 m^3 and 100 kPa, respectively.  Heat is transferred to the air at constant pressure until the volume is doubled. We can neglect kinetic and potential energy effects.  Given that the specific heat is constant, cv = 0.718 kJ/kg·K, and the specific gas constant R = 0.287 kJ/kg·K.  Calculate:  The heat transfer for the process. Determine: Heat transfer Q (in kJ for example).'),
  np.float32(1.051564)),
 (Document(id='8d0e0204-ee7f-48b4-8622-de9b7e52a8e6', metadata={'source': 'src\\data\\QuestionDataV2_06122025_classified.csv', 'index': 227, 'relevant_courses': 'ME 100A, ME 100B, ME 135', 'topics': 'Polytropic Processes, Ideal Gas Behavior'}, page_c

Inspecting the documetns

In [112]:
for d, score in docs[:2]:
    question = d.page_content
    topics = d.metadata.get("topics", "N/A")
    relevant_courses = d.metadata.get("relevant_courses", "N/A")
    print(
        f"Question: {question}\n\n {'-'*30}\nScore: {score}\nTopics: {topics}\nRelevant Courses: {relevant_courses}\n{'-'*40}"
    )

Question: 5 kg of air fills the cylinder of a piston-cylinder assembly. The initial volume and pressure are 1 m^3 and 100 kPa, respectively.  Heat is transferred to the air at constant pressure until the volume is doubled. We can neglect kinetic and potential energy effects.  Given that the specific heat is constant, cv = 0.718 kJ/kg·K, and the specific gas constant R = 0.287 kJ/kg·K.  Calculate:  The heat transfer for the process. Determine: Heat transfer Q (in kJ for example).

 ------------------------------
Score: 1.0515639781951904
Topics: Ideal Gas Behavior, Heat Transfer Area
Relevant Courses: ME 100A, ME 100B
----------------------------------------
Question: Within a piston-cylinder assembly, air undergoes three processes:  Process 1-2: Compression where PV = constant. Initial pressure and volume are 100 kPa and 0.5 m^3, respectively. The final pressure is 200 kPa.  Process 2-3: The process occurs at a constant volume where the pressure is 150 kPa.  Process 3-1: Constant press

In [122]:
docs = vectorstore.similarity_search(
    query="Thermo",
    k=3,
)
print(docs)

[Document(id='3ee3649e-5b51-4a8b-af1e-2b643a3cf6c8', metadata={'source': 'src\\data\\QuestionDataV2_06122025_classified.csv', 'index': 235, 'relevant_courses': 'ME 100B, ME 100A, ME 243', 'topics': 'Thermal Efficiency in Power Cycles, Heat Transfer'}, page_content='two reversible power cycles are in series aand each produce the same net work. \r\n the first cycle accepts energy $q_h$ from a hot reservoir that is at 500 k and reject energy q to a reservoir at an intermediate temperature t by heat transfer.\r\n the second cycle revceives energy q from the reservior at t and rejects energy $q_c$ to a reservoir at 300 k via heat transfer. \r\n\r\n determine the intermediate temperature and the thermal efficiencies.'), Document(id='75c63e1f-08f3-4965-a10b-f6f98e88c865', metadata={'source': 'src\\data\\QuestionDataV2_06122025_classified.csv', 'index': 202, 'relevant_courses': 'ME 116B, ME 241A', 'topics': 'Heat Transfer, Heat Transfer Area'}, page_content='determine the heat rate passing thr

We can add additinal filters such as filtering by course 
This can be expanded to other fields such as filtering by professors,topic etc. 

In [123]:
filter_q = {
    "relevant_courses": {"$in": ["ME100A", "ME 100B"]}
}
docs = vectorstore.similarity_search(
    query="Thermo",
    k=3,
    filter=filter_q,
)

## Generate Query


In [129]:
from langgraph.graph import MessagesState
from typing import Optional, List

response_model = ChatOpenAI(model="gpt-4o-mini")


def retrieve_questions(
    query: str,
    class_filter: Optional[List[str]] = None,
    search_results: Optional[int] = 3,
):
    """A search tool meant to find questions from a database to aid students takes in a query an optional
    class filter to filter based on class and number of results to bring"""

    # filter_q = {"relevant_courses": {"$in": class_filter}}
    filter_q = None

    documents = vectorstore.similarity_search_with_score(
        query=query,
        filter=filter_q,
        k=search_results if search_results is not None else 3,
    )

    return "/n".join([d.page_content for d, score in documents])





def generate_query_or_respond(state: MessagesState):
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    response = response_model.bind_tools([retrieve_questions]).invoke(state["messages"])
    return {"messages": [response]}

In [130]:
input = {"messages": [{"role": "user", "content": "hello!"}]}
generate_query_or_respond(input)["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello! How can I assist you today?


In [132]:
input = {"messages": [{"role": "user", "content": "hello I want to study some mechanical engineering question specifically I want to cover projectile motion questions. I want to practice 3 questions"}]}
generate_query_or_respond(input)["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  retrieve_questions (call_812bO9iaEMWQXmqFdYSjAND3)
 Call ID: call_812bO9iaEMWQXmqFdYSjAND3
  Args:
    query: projectile motion
    class_filter: ['mechanical engineering']
    search_results: 3


In [69]:
from langgraph.prebuilt import create_react_agent
from typing import Optional, List

model = ChatOpenAI(model="gpt-4o")


def retrieve_questions(
    query: str, class_filter: Optional[List[str]]=None, search_results: Optional[int] = 3
):
    """A search tool meant to find questions from a database to aid students takes in a query an optional
    class filter to filter based on class and number of results to bring"""

    documents = vectorstore.similarity_search_with_score(
        query=query,
        k=search_results if search_results is not None else 3,
    )
    

    return "/n".join([d.page_content for d, score in documents])


graph = create_react_agent(
    model,
    tools=[retrieve_questions],  # type: ignore
)
inputs = {
    "messages": [
        {
            "role": "user",
            "content": "I want to practice some thermodynamic courses the classes i am taking right now is me100A",
        }
    ]
}
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Axd1I6RoYebJQC2zcyw6pMIm', 'function': {'arguments': '{"query":"thermodynamics","class_filter":["me100A"],"search_results":5}', 'name': 'retrieve_questions'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 118, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-BhnHMoguddQq3u4umxtcgIGS1Qu76', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--ee1077f2-ec32-49f3-b8f9-f43c490125b4-0', tool_calls=[{'name': 'retrieve_questions', 'args': {'query': 'thermodynamics', 'class_filter': ['me100A'], 'search_results': 5}, 'id': 'call_Axd1I6RoY